In [54]:
import nltk
#nltk.download('averaged_perceptron_tagger_eng')
#nltk.download('punkt_tab')
#nltk.download('stopwords')

In [55]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
from pathlib import Path
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import contractions
from nltk import pos_tag
from tqdm import tqdm

In [56]:
from nltk.corpus import stopwords
from nltk import pos_tag
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [57]:
base_path_alvaro = Path(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project\TXT")

base_path = base_path_alvaro # change according to user

# 1. Prepare data

## Step 1: Run only once and make sure you have previously changed the files name of the first 9 files by taking the 0 out of it (out of the name).

In [58]:
sessions = np.arange(1, 10)

path = os.path.realpath(base_path)
path = os.startfile(path)

for session in sessions:
    year = session + 1945
    session_folder = base_path / f"Session {int(session):02d} - {year}"

    for file in session_folder.iterdir():
        new_name = file.stem.replace("0", "") + file.suffix
        new_path = file.with_name(new_name)
        file.rename(new_path)
        print(f"Renamed: {file.name} → {new_path.name}")
    


Renamed: ARG_1_1946.txt → ARG_1_1946.txt
Renamed: AUS_1_1946.txt → AUS_1_1946.txt
Renamed: BEL_1_1946.txt → BEL_1_1946.txt
Renamed: BLR_1_1946.txt → BLR_1_1946.txt
Renamed: BOL_1_1946.txt → BOL_1_1946.txt
Renamed: BRA_1_1946.txt → BRA_1_1946.txt
Renamed: CAN_1_1946.txt → CAN_1_1946.txt
Renamed: CHL_1_1946.txt → CHL_1_1946.txt
Renamed: CHN_1_1946.txt → CHN_1_1946.txt
Renamed: COL_1_1946.txt → COL_1_1946.txt
Renamed: CSK_1_1946.txt → CSK_1_1946.txt
Renamed: CUB_1_1946.txt → CUB_1_1946.txt
Renamed: ECU_1_1946.txt → ECU_1_1946.txt
Renamed: EGY_1_1946.txt → EGY_1_1946.txt
Renamed: FRA_1_1946.txt → FRA_1_1946.txt
Renamed: GBR_1_1946.txt → GBR_1_1946.txt
Renamed: GRC_1_1946.txt → GRC_1_1946.txt
Renamed: HTI_1_1946.txt → HTI_1_1946.txt
Renamed: IND_1_1946.txt → IND_1_1946.txt
Renamed: IRN_1_1946.txt → IRN_1_1946.txt
Renamed: LBN_1_1946.txt → LBN_1_1946.txt
Renamed: LBR_1_1946.txt → LBR_1_1946.txt
Renamed: LUX_1_1946.txt → LUX_1_1946.txt
Renamed: MEX_1_1946.txt → MEX_1_1946.txt
Renamed: NLD_1_1

## Step 2: Functions to count the number of tokens and sentences

In [59]:
def count_number_of_sentences_and_tokens(text):
    """Count the number of sentences and tokens in a text file."""
    number_sentences = len(sent_tokenize(text))
    number_tokens = len(word_tokenize(text))
    return number_sentences, number_tokens

## Step 3: Create DFs

In [60]:
dataset_path_alvaro = r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project"

dataset_path = dataset_path_alvaro # change according to user

### Check if thall the files are there

In [61]:
# Raw string path (exactly as shown in your File Explorer)
sessions = range(1, 80)
data = []

for session in tqdm(sessions):
    # Construct path using os.path.join() for reliability
    folder_name = f"Session {session:02d} - {1945 + session}"
    full_path = os.path.join(base_path, folder_name)
    
    # Check if path exists before using it
    if os.path.exists(full_path):
        data.append(full_path)
    else:
        print(f"Warning: Folder not found - {full_path}")

print(f"Found {len(data)} valid session folders")

100%|██████████| 79/79 [00:00<00:00, 27420.56it/s]

Found 79 valid session folders


In [62]:
def load(url):
    with open(url, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

## Step 4: DF raw Run Once

In [63]:
sessions = np.arange(1, 80)
data=[]

for session in tqdm(sessions):
    # directory = base_path + "\\Session "+str(0 + session)+" - "+str(1945+session)
    directory = f"{base_path}\\Session {session:02d} - {1945 + session}"
    for filename in os.listdir(directory):
        if filename[0] == ".":  # Skip hidden files
            continue
        filepath = os.path.join(directory, filename)
        splt = filename.split('_')
        txt = load(f'{filepath}')
        number_sentences, number_tokens = count_number_of_sentences_and_tokens(txt)
        data.append([session, 1945+session, splt[0], txt, number_sentences, number_tokens])
        
df_raw = pd.DataFrame(data, columns=['Session','Year','ISO-Code','Speech', "number_sentences", "number_tokens"])
df_raw.to_csv(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project" + "\\UN_Speeches_raw.csv", index=False, encoding='utf-8')
df_raw

100%|██████████| 79/79 [03:21<00:00,  2.55s/it]


,Session,Year,ISO-Code,Speech,number_sentences,number_tokens
0,1,1946,ARG,At the resumption of the first session of the ...,126,3739
1,1,1946,AUS,The General Assembly of the United Nations is ...,124,4897
2,1,1946,BEL,The\tprincipal organs of the United Nations ha...,97,2750
3,1,1946,BLR,As more than a year has elapsed since the Unit...,93,3350
4,1,1946,BOL,Coming to this platform where so many distingu...,52,1686
...,...,...,...,...,...,...
10965,79,2024,WSM,"Excellencies, \nI extend my congratulations t...",68,1572
10966,79,2024,YEM,"Your Majesties, Excellencies, and Highnesses, ...",57,1876
10967,79,2024,ZAF,President of the 79th Session of the UN Genera...,100,1870
10968,79,2024,ZMB,"\n YOUR EXCELLENCY PHILEMON YANG, PRESIDENT O...",81,2348


# 2. Join with the World bank label data

In [64]:
df_raw = pd.read_csv(dataset_path + "\\UN_Speeches_raw.csv") 
df_raw_1990 = df_raw[df_raw['Year'] >= 1990]
df_raw_1990

,Session,Year,ISO-Code,Speech,number_sentences,number_tokens
4430,45,1990,AFG,"﻿Allow me, first of all, Sir, to congratulate ...",159,4982
4431,45,1990,AGO,"﻿First I would like to congratulate you, Sir, ...",77,2970
4432,45,1990,ALB,﻿It is a special pleasure for me to speak at t...,112,3783
4433,45,1990,ARE,"﻿\nMr. President, on behalf of the delegation ...",115,3407
4434,45,1990,ARG,"﻿At the outset, let me convey to you, Sir, my ...",81,2816
...,...,...,...,...,...,...
10965,79,2024,WSM,"Excellencies, \nI extend my congratulations t...",68,1572
10966,79,2024,YEM,"Your Majesties, Excellencies, and Highnesses, ...",57,1876
10967,79,2024,ZAF,President of the 79th Session of the UN Genera...,100,1870
10968,79,2024,ZMB,"\n YOUR EXCELLENCY PHILEMON YANG, PRESIDENT O...",81,2348


In [65]:
label_income_1990 = pd.read_csv(dataset_path + "label_world_bank_1990.csv") 
label_income_1990

,Entity,ISO-Code,Year,Income Level
0,Afghanistan,AFG,1990,1.0
1,Afghanistan,AFG,1991,1.0
2,Afghanistan,AFG,1992,1.0
3,Afghanistan,AFG,1993,1.0
4,Afghanistan,AFG,1994,1.0
...,...,...,...,...
7325,Virgin Islands (U.S.),VIR,2024,4.0
7326,West Bank and Gaza,PSE,2024,2.0
7327,"Yemen, Rep.",YEM,2024,1.0
7328,Zambia,ZMB,2024,2.0


In [66]:
df_speeches_1990_code_list = df_raw_1990['ISO-Code'].unique() 
wb_labels_final_1990_list = label_income_1990['ISO-Code'].unique()

not_in_speeches = [item for item in wb_labels_final_1990_list if item not in df_speeches_1990_code_list]
print("Country codes in WB, but not in UN speeches:", not_in_speeches)

not_in_wb = [item for item in df_speeches_1990_code_list if item not in wb_labels_final_1990_list]
print("Country codes in UN speeches, but not in WB:", not_in_wb)


Country codes in WB, but not in UN speeches: ['ASM', 'ABW', 'BMU', 'VGB', 'CYM', 'CHI', 'CUW', 'FRO', 'PYF', 'GIB', 'GRL', 'GUM', 'HKG', 'IMN', 'XKX', 'MAC', 'NCL', 'MNP', 'PRI', 'MAF', 'SXM', 'TWN', 'TCA', 'VIR']
Country codes in UN speeches, but not in WB: ['CSK', 'DDR', 'VAT', 'EU']


In [67]:
def add_income_level(speeches_df, income_df):
    # merge on ISO-Code and Year
    merged = speeches_df.merge(
        income_df[['ISO-Code', 'Year', 'Income Level']],
        how='left',
        on=['ISO-Code', 'Year']
    )
    
    # fill missing income levels with 0
    merged['Income Level'] = merged['Income Level'].fillna(0).astype(int)
    
    return merged

df_combined = add_income_level(df_raw_1990, label_income_1990)
df_combined

,Session,Year,ISO-Code,Speech,number_sentences,number_tokens,Income Level
0,45,1990,AFG,"﻿Allow me, first of all, Sir, to congratulate ...",159,4982,1
1,45,1990,AGO,"﻿First I would like to congratulate you, Sir, ...",77,2970,2
2,45,1990,ALB,﻿It is a special pleasure for me to speak at t...,112,3783,2
3,45,1990,ARE,"﻿\nMr. President, on behalf of the delegation ...",115,3407,4
4,45,1990,ARG,"﻿At the outset, let me convey to you, Sir, my ...",81,2816,2
...,...,...,...,...,...,...,...
6535,79,2024,WSM,"Excellencies, \nI extend my congratulations t...",68,1572,2
6536,79,2024,YEM,"Your Majesties, Excellencies, and Highnesses, ...",57,1876,1
6537,79,2024,ZAF,President of the 79th Session of the UN Genera...,100,1870,3
6538,79,2024,ZMB,"\n YOUR EXCELLENCY PHILEMON YANG, PRESIDENT O...",81,2348,2


In [68]:
list_not_in_WB = df_combined[df_combined['Income Level'] == 0]['ISO-Code'].unique()
list_not_in_WB

array(['BLR', 'CSK', 'DDR', 'LIE', 'RUS', 'UKR', 'YUG', 'MCO', 'SMR',
       'NRU', 'TUV', 'VAT', 'EU', 'VEN'], dtype=object)

**Notes in the results above**

TUV -> Tuvalu 

NRU -> Nauru

SMR -> San Marino

VAT -> Vatican is an observer state 

EU -> Overall represenatative

VEN -> Venezuela Not enough data

In [69]:
df_combined[df_combined['ISO-Code'] == 'DDR']

,Session,Year,ISO-Code,Speech,number_sentences,number_tokens,Income Level
38,45,1990,DDR,"﻿Mr. President, I wish to thank you on behalf ...",24,539,0


# 3. Final DF

In [70]:
final_df = df_combined[df_combined['Income Level'] != 0]
final_df

,Session,Year,ISO-Code,Speech,number_sentences,number_tokens,Income Level
0,45,1990,AFG,"﻿Allow me, first of all, Sir, to congratulate ...",159,4982,1
1,45,1990,AGO,"﻿First I would like to congratulate you, Sir, ...",77,2970,2
2,45,1990,ALB,﻿It is a special pleasure for me to speak at t...,112,3783,2
3,45,1990,ARE,"﻿\nMr. President, on behalf of the delegation ...",115,3407,4
4,45,1990,ARG,"﻿At the outset, let me convey to you, Sir, my ...",81,2816,2
...,...,...,...,...,...,...,...
6535,79,2024,WSM,"Excellencies, \nI extend my congratulations t...",68,1572,2
6536,79,2024,YEM,"Your Majesties, Excellencies, and Highnesses, ...",57,1876,1
6537,79,2024,ZAF,President of the 79th Session of the UN Genera...,100,1870,3
6538,79,2024,ZMB,"\n YOUR EXCELLENCY PHILEMON YANG, PRESIDENT O...",81,2348,2


# 4. Clean Without Postagging

In [71]:
def take_out_mentions_to_the_president(text):
    """Remove introductory phrases addressing the president or other dignitaries from text.
    Args:
        text (str): The input text to process    
    Returns:
        str: The text with introductory address phrases removed, or original text if none found
    """
    
    # Split into sentences (simple regex)
    # The regex looks for sentence-ending punctuation followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    
    # Get first 4 sentences as search area - addresses are typically at the beginning
    search_area = ' '.join(sentences[:2]).lower()
    
    # Phrases to search for (lowercase)
    # We look for various forms of presidential/dignitary addresses
    # Store the index where each phrase is found (-1 if not found)
    possible_starts = [
        search_area.lower().find("mr. president"),  # Formal address with period
        search_area.lower().find("mr president"),   # Formal address without period
        search_area.lower().find("ladies and gentlemen"),  # Common formal address
        search_area.lower().find("your excellency"),  # Covers "your excellence" and "your excellency"
        search_area.lower().find("your excellence"),  # Covers "your excellence" and "your excellency"
        search_area.lower().find("your majesties"),  # 
        search_area.lower().find("your majesties"),  # 
        search_area.lower().find("president of the general assembly"),
        search_area.lower().find("excellencies"),     # Plural form of address
        search_area.lower().find("esteemed president"),  # Honorific address
        search_area.lower().find("distinguished delegates"),  # Assembly address
        search_area.lower().find("allah"),  # Address to members of an assembly
    ]
    
    # Filter out phrases that weren't found (-1) and keep only valid indices
    starts = [start for start in possible_starts if start != -1]
    
    # Get the latest occurrence of any address phrase
    start = max(starts) if len(starts) > 0 else -1
    
    if start == -1:
        # No address phrases found - log this and return full text
        # print(f"Could not find probable start in the text of {url}")
        start = 0
    
    # Return text starting from after the address phrase
    return text[start:]

In [72]:
def simple_clean(text):
    """Cleans text by removing common formatting artifacts from PDF conversions and UN document patterns.
    
    Args:
        text (str): Input text to be cleaned
        
    Returns:
        str: Cleaned text with unwanted patterns removed
    """
    
    # Convert to lowercase for consistent processing
    text_1 = text.lower()
  
    # 1: Remove UN document reference numbers (e.g. "20/26 15-29876")
    text = re.sub(r'\b\d{1,4}\s*/\s*\d{1,4}\s+\d{2,4}-\d{4,8}\b', ' ', text_1)
    
    # 2: Remove UN meeting record references (e.g. "A/70/PV.24")
    text = re.sub(r'\b[a-z]\s*/\s*\d+\s*/pv\s*\.\s*\d+\b', ' ', text, flags=re.IGNORECASE)
    
    # 3: Remove dates in DD/MM/YYYY format (e.g. "31/12/2023")
    text = re.sub(r'\d{2}\/\d{2}\/\d{4}', '', text) 
  
    # 4: Remove form feed characters (often from PDF conversion)
    text = re.sub(r'\x0c', '', text) 
  
    # 5: Remove parenthetical document references (e.g. "(A/70/123)" or "(A/70/123, annex)")
    text = re.sub(r'\(\s*[a-z]\s*/\s*\d+\s*/\s*\d+\s*(?:,\s*annex)?\s*\)', '', text, flags=re.IGNORECASE)
    
    # 6: Remove standalone line numbers/page numbers (e.g. "42" on its own line)
    text = re.sub(r'^\s*\d+\s*$\n?', '', text, flags=re.MULTILINE)
    
    # 7: Normalize newlines - replace all with single spaces
    text = re.sub(r'\n', ' ', text, flags=re.MULTILINE)

    # 8: Remove numbered list prefixes (e.g. "1.    Some text")
    text = re.sub(r'\d+\.\t', '', text)

    # 9: Remove Unicode BOM (Byte Order Mark) character if present
    text = text.replace('\ufeff', '')
    
    # 10: Normalize whitespace - collapse multiple spaces into one and trim
    text = re.sub(r'\s+', ' ', text).strip()

    # 11. Remove standalone hyphens
    text = re.sub(r'\s*-(?!\w)(?<!\w)-*\s*', ' ', text)

    # 12: Remove symbols
    text = text.replace('—', '').replace(',', '').replace(':', '').replace('’', '').replace('“', '').replace("”", '').replace(";", '').replace("''", '')
    
    # 12: Remove words/numbers between parenthesis
    text = re.sub(r'\([^)]*\)', '', text)

    #13: Remove markdown-style bold/italic/blockquote symbols (all below are subpoints rather that main ones)
    text = re.sub(r'\*\*+', '', text)       # removes **, **** etc.
    text = re.sub(r'>+', '', text)          # removes >, >>, etc.
    text = re.sub(r'-{2,}', '', text)       # removes --, --- etc.
    text = re.sub(r'[=*_~#`]+', '', text)   # removes *, _, =, #, ~, ` etc.
    # I want to replace this simbol • with a space 
    text = text.replace('•', ' ')            # replaces • with a space

    # 14: Remove \n+ (multiple newlines) and replace with a single space
    text = re.sub(r'\n+', ' ', text)

    # 15: Remove IDs like: "123 84-98765", "84-9876 56", "1234 12-34567"
    pattern = r'\b(?:\d{1,4}\s+)?\d{1,4}-\d{4,}(?:\s+\d{1,4})?\b'
    text = re.sub(pattern, '', text)

    # 16: Remove any remaining leading/trailing whitespace
    text = re.sub(r'\s{2,}', ' ', text).strip()

    return text

In [73]:
def remove_stopwords(text):
    """Remove common stopwords from text"""
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text)
    filtered_words = [word for word in words 
                 if word.lower() not in stop_words and len(word) > 2]
    return ' '.join(filtered_words)                          

In [74]:
def remove_punctuations(text):
    return text.replace('.', '').replace('?', '').replace('!', '')

# 5. Clean with Postagging

In [75]:
stop_words = set(stopwords.words('english'))

In [76]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
# Initialize stemmer/lemmatizer (run once)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

## CLEAN THE WHOLE SPEECH

In [77]:
# Assuming these are defined elsewhere or should be imported
# from your_module import simple_clean, take_out_mentions_to_the_president
# from your_module import stop_words  # or define here

def filter_by_pos(text):
    """Filter words based on their part-of-speech tags.
    
    Args:
        text (str): Input text to filter
        
    Returns:
        tuple: (filtered_words, kept_tags) - lists of words and their POS tags
    """
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    
    keep_pos = {'NN','NNS','NNP','NNPS','JJ','JJR','JJS',
                'VB','VBD','VBG','VBN','VBP','VBZ',
                'RB','RBR','RBS', 'RP', 'MD'}
    
    filtered = []
    kept_tags = []
    for word, tag in pos_tags:
        if tag in keep_pos:
            filtered.append(word)
            kept_tags.append(tag)
    
    return filtered, kept_tags

def get_wordnet_pos(treebank_tag):
    """Convert POS tag to WordNet format.
    
    Args:
        treebank_tag (str): POS tag in Treebank format
        
    Returns:
        str: WordNet POS tag
    """
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:  # Default to noun
        return wordnet.NOUN

def clean_postagging(text):
    """Clean text with POS-aware lemmatization without re-tagging."""
    # Step 1: Fix contractions and clean text
    text = contractions.fix(text)
    text = simple_clean(text)  
    text = take_out_mentions_to_the_president(text)  

    # Step 2: POS filtering + get original tags
    filtered_words, filtered_pos_tags = filter_by_pos(text)  

    # Step 3: Clean words and align with POS tags
    processed_words = []
    for word, tag in zip(filtered_words, filtered_pos_tags):
        # Remove punctuation and validate
        word_clean = re.sub(r'[^\w\s]', '', word)
        if word_clean and len(word_clean) > 2 and word_clean.lower() not in stop_words:
            # Lemmatize with original POS tag
            lemma = lemmatizer.lemmatize(word_clean, pos=get_wordnet_pos(tag))
            processed_words.append(lemma)

    return ' '.join(processed_words)

In [78]:
tqdm.pandas()  # Enable progress_apply in pandas

final_df['cleaned_speeches_postagging_expanded'] = final_df['Speech'].progress_apply(
    lambda x: clean_postagging(
        x
    )
)
final_df

100%|██████████| 6457/6457 [07:47<00:00, 13.81it/s]
C:\Users\alvar\AppData\Local\Temp\ipykernel_12516\3277632493.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['cleaned_speeches_postagging_expanded'] = final_df['Speech'].progress_apply(


,Session,Year,ISO-Code,Speech,number_sentences,number_tokens,Income Level,cleaned_speeches_postagging_expanded
0,45,1990,AFG,"﻿Allow me, first of all, Sir, to congratulate ...",159,4982,1,allow first sir congratulate unanimous electio...
1,45,1990,AGO,"﻿First I would like to congratulate you, Sir, ...",77,2970,2,first would like congratulate sir election pre...
2,45,1990,ALB,﻿It is a special pleasure for me to speak at t...,112,3783,2,special pleasure speak session general assembl...
3,45,1990,ARE,"﻿\nMr. President, on behalf of the delegation ...",115,3407,4,president behalf delegation united arab emirat...
4,45,1990,ARG,"﻿At the outset, let me convey to you, Sir, my ...",81,2816,2,president general assembly fortyfifth session ...
...,...,...,...,...,...,...,...,...
6535,79,2024,WSM,"Excellencies, \nI extend my congratulations t...",68,1572,2,excellency extend congratulation excellency ph...
6536,79,2024,YEM,"Your Majesties, Excellencies, and Highnesses, ...",57,1876,1,lady gentleman happy coincidence address today...
6537,79,2024,ZAF,President of the 79th Session of the UN Genera...,100,1870,3,president session general assembly philemon ya...
6538,79,2024,ZMB,"\n YOUR EXCELLENCY PHILEMON YANG, PRESIDENT O...",81,2348,2,lady gentleman congratulate excellency assumpt...


## CLEAN ONLY ENOUGH SO SENTENCES CAN BE RETREIVED

In [79]:
def clean_speeches(text, use_simple_clean=True, disregard_mentions_to_president=True, expand=False):
    """Load a text file and apply cleaning operations. 
    Args:
        text: Original text
        use_simple_clean (bool): Whether to apply basic text cleaning
        disregard_mentions_to_president (bool): Whether to remove mentions to the president
        Returns:
        str: Cleaned text from the file"""
    
    if expand: 
        text = contractions.fix(text) 

    # Apply basic text cleaning if the flag is set
    if use_simple_clean:
        text = simple_clean(text)  # Assumes simple_clean is a custom function defined elsewhere

    # Remove mentions to the president if the flag is set
    if disregard_mentions_to_president:
        text = take_out_mentions_to_the_president(text)  # Also assumes this function is defined elsewhere

    # Return the cleaned (or original) text
    return text

In [80]:
tqdm.pandas()  # Enable pandas integration

final_df['speeches_for_keyword_search'] = final_df['Speech'].progress_apply(
    lambda x: clean_speeches(
        x, 
        expand=True,
        use_simple_clean=True, 
        disregard_mentions_to_president=True
    )
)

final_df 


100%|██████████| 6457/6457 [00:22<00:00, 289.23it/s]
C:\Users\alvar\AppData\Local\Temp\ipykernel_12516\3522202045.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['speeches_for_keyword_search'] = final_df['Speech'].progress_apply(


,Session,Year,ISO-Code,Speech,number_sentences,number_tokens,Income Level,cleaned_speeches_postagging_expanded,speeches_for_keyword_search
0,45,1990,AFG,"﻿Allow me, first of all, Sir, to congratulate ...",159,4982,1,allow first sir congratulate unanimous electio...,allow me first of all sir to congratulate you ...
1,45,1990,AGO,"﻿First I would like to congratulate you, Sir, ...",77,2970,2,first would like congratulate sir election pre...,first i would like to congratulate you sir on ...
2,45,1990,ALB,﻿It is a special pleasure for me to speak at t...,112,3783,2,special pleasure speak session general assembl...,it is a special pleasure for me to speak at th...
3,45,1990,ARE,"﻿\nMr. President, on behalf of the delegation ...",115,3407,4,president behalf delegation united arab emirat...,mr. president on behalf of the delegation of t...
4,45,1990,ARG,"﻿At the outset, let me convey to you, Sir, my ...",81,2816,2,president general assembly fortyfifth session ...,president of the general assembly at its forty...
...,...,...,...,...,...,...,...,...,...
6535,79,2024,WSM,"Excellencies, \nI extend my congratulations t...",68,1572,2,excellency extend congratulation excellency ph...,excellencies i extend my congratulations to hi...
6536,79,2024,YEM,"Your Majesties, Excellencies, and Highnesses, ...",57,1876,1,lady gentleman happy coincidence address today...,ladies and gentlemen it is a happy coincidence...
6537,79,2024,ZAF,President of the 79th Session of the UN Genera...,100,1870,3,president session general assembly philemon ya...,president of the 79th session of the un genera...
6538,79,2024,ZMB,"\n YOUR EXCELLENCY PHILEMON YANG, PRESIDENT O...",81,2348,2,lady gentleman congratulate excellency assumpt...,ladies and gentlemen i congratulate you your e...


In [ ]:
climate_keywords = ["climate change", "global warming", "global warm","cap and trade", "paris accord", "emissions trading", "global average temperature", 
                      "kyoto protocol", "changing climate" ,"climate resilience","climate decay", "carbon dioxide", "carbon-dioxide","climate politics", 
                      "framework convention on climate change", "bali roadmap", "bali action plan", 
                      "greenhouse gas", "greenhouse-gas","greenhouse effect", "climate mitigation", "climate action", "emissions", "temperature", "extreme weather", 
                      "global environmental change", "global environment", "global environmental" ,"climate variability", "low carbon", "renewable energy", 
                      "carbon emission", "climate pollutant", "climate pollutants", "carbon tax", "carbon footprint", "carbon neutrality", "net-zero", 
                      "net zero","net-zero","climate crisis", "climate summit", "climate catastrophe", "climate justice", "climate emergency", "climate funding",
                      "climate fund", "climate financing", "climate finance","climate peace", "climate agreement", "climate security", "climate ambition", 
                      "climate issue", "climate impact", "climate conference", "climate event", "climate challenge", "climate trust", "climate negotiation", 
                      "climate catastrophe", "climate risk", "climate goal", "climate change-related", "climate regime", "climate resilient", "climate policy", 
                      "carbon market", "carbon sink", "green climate", "green economy", "emission reduction", "emissions reduction", "carbon neutral", 
                      "ozone layer", "unfccc", "emissions trading scheme", "ghg", "ghge", "co2", "co2 emission", "ipcc", "decarbonisation", "decarbonization"]

In [82]:
def extract_keyword_info(text, keywords):
    """
    Extracts climate-related keyword information from a text string.

    Parameters:
        text (str): The speech or text in which to search for keywords.
        keywords (list of str): A list of climate-related keywords or phrases to match against the text.
    """
    matches = []
    for kw in keywords:
        pattern = r'\b' + re.escape(kw) + r's?\b' # Match also plural forms
        if re.search(pattern, text, flags=re.IGNORECASE):
            matches.append(kw)
    contains_keyword = len(matches) > 0
    return pd.Series([matches, contains_keyword])



final_df[['matched_climate_keywords', 'contains_climate_keyword']] = final_df['speeches_for_keyword_search'].progress_apply(
    lambda text: extract_keyword_info(text, climate_keywords)
)

100%|██████████| 6457/6457 [02:00<00:00, 53.54it/s]
C:\Users\alvar\AppData\Local\Temp\ipykernel_12516\279498772.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df[['matched_climate_keywords', 'contains_climate_keyword']] = final_df['speeches_for_keyword_search'].progress_apply(
C:\Users\alvar\AppData\Local\Temp\ipykernel_12516\279498772.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df[['matched_climate_keywords', 'contains_climate_keyword']] = final_df['speeches_for_keyword_search'].

In [83]:
final_df

,Session,Year,ISO-Code,Speech,number_sentences,number_tokens,Income Level,cleaned_speeches_postagging_expanded,speeches_for_keyword_search,matched_climate_keywords,contains_climate_keyword
0,45,1990,AFG,"﻿Allow me, first of all, Sir, to congratulate ...",159,4982,1,allow first sir congratulate unanimous electio...,allow me first of all sir to congratulate you ...,[],False
1,45,1990,AGO,"﻿First I would like to congratulate you, Sir, ...",77,2970,2,first would like congratulate sir election pre...,first i would like to congratulate you sir on ...,[],False
2,45,1990,ALB,﻿It is a special pleasure for me to speak at t...,112,3783,2,special pleasure speak session general assembl...,it is a special pleasure for me to speak at th...,[],False
3,45,1990,ARE,"﻿\nMr. President, on behalf of the delegation ...",115,3407,4,president behalf delegation united arab emirat...,mr. president on behalf of the delegation of t...,[],False
4,45,1990,ARG,"﻿At the outset, let me convey to you, Sir, my ...",81,2816,2,president general assembly fortyfifth session ...,president of the general assembly at its forty...,[],False
...,...,...,...,...,...,...,...,...,...,...,...
6535,79,2024,WSM,"Excellencies, \nI extend my congratulations t...",68,1572,2,excellency extend congratulation excellency ph...,excellencies i extend my congratulations to hi...,"[climate change, climate action, temperature, ...",True
6536,79,2024,YEM,"Your Majesties, Excellencies, and Highnesses, ...",57,1876,1,lady gentleman happy coincidence address today...,ladies and gentlemen it is a happy coincidence...,[climate change],True
6537,79,2024,ZAF,President of the 79th Session of the UN Genera...,100,1870,3,president session general assembly philemon ya...,president of the 79th session of the un genera...,"[climate change, climate action, emissions, ex...",True
6538,79,2024,ZMB,"\n YOUR EXCELLENCY PHILEMON YANG, PRESIDENT O...",81,2348,2,lady gentleman congratulate excellency assumpt...,ladies and gentlemen i congratulate you your e...,"[climate change, extreme weather, climate fina...",True


In [84]:
final_df.to_csv(r"C:\Users\alvar\OneDrive\Escritorio\BDS\Block_5\NLP\Project" + "\\Final_df.csv", index=False, encoding='utf-8')